# Milestone 3: EDA dan Insight

Analisis temporal dan spasial pada **Indonesian News Dataset** yang sudah dibersihkan (`data/processed/dataset_bersih.parquet`, hasil Milestone 2).

- **DuckDB (SQL)** dipakai untuk agregasi langsung pada file Parquet.
- **Polars** dipakai untuk transformasi (teks judul, penyebutan wilayah, pivot manual).
- **Plotly** dipakai untuk 8 visualisasi interaktif. Setiap grafik disimpan ke `output/figures/` sebagai HTML.

> **Catatan keterbatasan:** dataset ini hasil *scraping* situs berita. Jumlah artikel per sumber dan per bulan mencerminkan cakupan *crawler*, belum tentu volume terbit yang sebenarnya. Bulan pertama dan terakhir juga mungkin tidak lengkap.

## 0. Konfigurasi dan Import

In [1]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import duckdb
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

pio.templates.default = "plotly_white"
# Jika grafik tidak tampil di VS Code, buka komentar baris berikut:
pio.renderers.default = "notebook_connected"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":      # notebook dijalankan dari folder notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
FIG_DIR = PROJECT_ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Data bersih tidak ditemukan: {DATA_PATH}. Jalankan 02_data_cleaning.ipynb sampai selesai."
    )

PARQUET = DATA_PATH.as_posix()


def simpan(fig, nama):
    """Simpan grafik interaktif ke output/figures/ sebagai HTML."""
    path = FIG_DIR / f"{nama}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Tersimpan:", path.relative_to(PROJECT_ROOT))


try:
    connection.close()
except Exception:
    pass
connection = duckdb.connect()


def query_pl(sql: str) -> pl.DataFrame:
    """Jalankan query DuckDB dan kembalikan hasilnya sebagai DataFrame Polars."""
    rel = connection.sql(sql)
    try:
        return rel.pl()
    except AttributeError:
        for nama in ("to_arrow_table", "fetch_arrow_table", "arrow"):
            if hasattr(rel, nama):
                return pl.from_arrow(getattr(rel, nama)())
        raise

## 1. Memuat Data

Hanya kolom yang dibutuhkan yang dimuat. Kolom `content` dan `summary` (teks panjang) tidak diperlukan sehingga memori tetap hemat.

In [3]:
df = pl.read_parquet(
    DATA_PATH,
    columns=["id", "source", "title", "date", "panjang_isi", "outlier_panjang_isi"],
)
print(f"Data: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print("Rentang tanggal:", df["date"].min(), "s.d.", df["date"].max())
df.head()

Data: 32,273 baris x 6 kolom
Rentang tanggal: 2018-08-10 s.d. 2023-04-11


id,source,title,date,panjang_isi,outlier_panjang_isi
i64,str,str,date,u32,bool
83,"""tempo""","""Depo Plumpang Terbakar, Anggot…",2023-03-04,2110,false
84,"""tempo""","""Jokowi Perintahkan Wapres Ma'r…",2023-03-04,2436,false
85,"""tempo""","""HNW Mendukung Jamaah Umroh Fir…",2023-03-04,2788,false
86,"""tempo""","""Tim Dokkes Polri Telah Terima …",2023-03-04,1981,false
87,"""tempo""","""Bamsoet Ajak Komunitas Otomoti…",2023-03-04,3471,false


## 2. Ringkasan per Sumber Berita (DuckDB)

In [4]:
ringkasan_sumber = query_pl(f"""
    SELECT source,
           COUNT(*) AS jumlah_artikel,
           ROUND(AVG(panjang_isi), 0) AS rata_panjang_isi,
           MIN("date") AS tanggal_awal,
           MAX("date") AS tanggal_akhir
    FROM read_parquet('{PARQUET}')
    GROUP BY source
    ORDER BY jumlah_artikel DESC
""")
ringkasan_sumber

source,jumlah_artikel,rata_panjang_isi,tanggal_awal,tanggal_akhir
str,i64,f64,date,date
"""kumparan""",8569,2131.0,2023-03-08,2023-04-11
"""okezone""",7063,2046.0,2023-03-11,2023-04-11
"""cnbcindonesia""",4498,2346.0,2023-03-08,2023-04-11
"""suara""",4004,2478.0,2023-03-12,2023-04-11
"""tempo""",3642,3042.0,2023-03-03,2023-04-11
"""cnnindonesia""",3488,2153.0,2018-08-10,2023-04-11
"""jawapos""",1009,2383.0,2023-03-10,2023-03-29


### Visualisasi 1: Jumlah artikel per sumber

In [5]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install -U nbformat ipython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
print(sys.executable)
try:
    import nbformat, IPython
    print("nbformat", nbformat.__version__, "| ipython", IPython.__version__)
except Exception as e:
    print("Belum ada:", e)

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\python.exe
nbformat 5.11.1 | ipython 9.18.1


In [8]:
sumber = ringkasan_sumber["source"].to_list()
jumlah = ringkasan_sumber["jumlah_artikel"].to_list()

fig1 = go.Figure(go.Bar(x=sumber, y=jumlah, text=[f"{j:,}" for j in jumlah], textposition="outside"))
fig1.update_layout(
    title="Jumlah Artikel per Sumber Berita",
    xaxis_title="Sumber", yaxis_title="Jumlah artikel",
)
simpan(fig1, "01_artikel_per_sumber")
fig1.show()

Tersimpan: output\figures\01_artikel_per_sumber.html


## 3. Analisis Temporal

### Visualisasi 2: Tren jumlah artikel per bulan

In [9]:
per_bulan = query_pl(f"""
    SELECT date_trunc('month', "date") AS bulan, COUNT(*) AS jumlah_artikel
    FROM read_parquet('{PARQUET}')
    GROUP BY 1
    ORDER BY 1
""")

fig2 = go.Figure(go.Scatter(
    x=per_bulan["bulan"].to_list(), y=per_bulan["jumlah_artikel"].to_list(),
    mode="lines+markers", name="Total artikel",
))
fig2.update_layout(
    title="Jumlah Artikel per Bulan (semua sumber)",
    xaxis_title="Bulan", yaxis_title="Jumlah artikel", hovermode="x unified",
)
simpan(fig2, "02_tren_artikel_per_bulan")
fig2.show()

Tersimpan: output\figures\02_tren_artikel_per_bulan.html


### Visualisasi 3: Tren bulanan per sumber

In [10]:
per_bulan_sumber = query_pl(f"""
    SELECT date_trunc('month', "date") AS bulan, source, COUNT(*) AS jumlah_artikel
    FROM read_parquet('{PARQUET}')
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

fig3 = go.Figure()
for nama in sumber:
    bagian = per_bulan_sumber.filter(pl.col("source") == nama)
    fig3.add_trace(go.Scatter(
        x=bagian["bulan"].to_list(), y=bagian["jumlah_artikel"].to_list(),
        mode="lines+markers", name=nama,
    ))
fig3.update_layout(
    title="Jumlah Artikel per Bulan menurut Sumber",
    xaxis_title="Bulan", yaxis_title="Jumlah artikel", hovermode="x unified",
)
simpan(fig3, "03_tren_per_sumber")
fig3.show()

Tersimpan: output\figures\03_tren_per_sumber.html


### Visualisasi 4: Pola hari terbit per sumber (heatmap)

Nilai adalah persentase artikel suatu sumber yang terbit pada hari tertentu, sehingga antar sumber bisa dibandingkan.

In [11]:
hari_nama = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]

per_hari = (
    df.with_columns(pl.col("date").dt.weekday().alias("hari"))   # 1 = Senin ... 7 = Minggu
    .group_by(["source", "hari"])
    .agg(pl.len().alias("n"))
)
total_sumber = dict(zip(*per_hari.group_by("source").agg(pl.col("n").sum()).to_dict(as_series=False).values()))
nilai = {(r["source"], r["hari"]): r["n"] for r in per_hari.iter_rows(named=True)}

z = [
    [round(nilai.get((s, h), 0) / total_sumber[s] * 100, 2) for h in range(1, 8)]
    for s in sumber
]

fig4 = go.Figure(go.Heatmap(
    z=z, x=hari_nama, y=sumber, colorscale="Blues",
    colorbar_title="% artikel", hovertemplate="%{y}<br>%{x}: %{z}%<extra></extra>",
))
fig4.update_layout(title="Persentase Artikel menurut Hari Terbit per Sumber")
simpan(fig4, "04_heatmap_hari_sumber")
fig4.show()

Tersimpan: output\figures\04_heatmap_hari_sumber.html


## 4. Karakteristik Artikel

### Visualisasi 5: Distribusi panjang isi artikel per sumber

Sumbu-y memakai skala logaritmik karena panjang artikel sangat menceng. Untuk menjaga ukuran file HTML, tiap sumber disampel maksimal 20.000 artikel.

In [12]:
fig5 = go.Figure()
for nama in sumber:
    bagian = df.filter(pl.col("source") == nama)
    contoh = bagian.sample(n=min(20_000, bagian.height), seed=42)
    fig5.add_trace(go.Box(y=contoh["panjang_isi"].to_list(), name=nama, boxpoints=False))
fig5.update_layout(
    title="Distribusi Panjang Isi Artikel per Sumber (sampel)",
    yaxis_title="Panjang isi (karakter, skala log)", yaxis_type="log", showlegend=False,
)
simpan(fig5, "05_boxplot_panjang_isi")
fig5.show()

Tersimpan: output\figures\05_boxplot_panjang_isi.html


## 5. Analisis Topik dari Judul

Judul dinormalisasi (huruf kecil, hanya huruf), dipecah menjadi kata, lalu kata umum (*stopword*) dan kata pendek dibuang.

In [13]:
STOPWORDS = {
    "yang", "dan", "dari", "untuk", "dengan", "pada", "dalam", "akan", "atau", "juga", "tidak",
    "oleh", "karena", "saat", "usai", "soal", "bisa", "jadi", "sebagai", "adalah", "hingga",
    "lebih", "para", "sudah", "telah", "masih", "kata", "setelah", "antara", "agar", "tanpa",
    "bakal", "hanya", "mereka", "kami", "kita", "anda", "saya", "bagi", "lalu", "kembali",
    "serta", "namun", "sampai", "seperti", "tentang", "apa", "siapa", "kenapa", "bagaimana",
    "ini", "itu", "ada", "lagi", "baru", "terbaru", "hari", "video", "foto", "berita",
}

kata_df = (
    df.select(
        pl.col("date"),
        pl.col("title").str.to_lowercase().str.replace_all(r"[^a-z\s]", " ").str.split(" ").alias("kata"),
    )
    .explode("kata")
    .filter((pl.col("kata").str.len_chars() >= 4) & (~pl.col("kata").is_in(list(STOPWORDS))))
)

top_kata = (
    kata_df.group_by("kata").agg(pl.len().alias("frekuensi"))
    .sort("frekuensi", descending=True).head(20)
)
top_kata

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17716\3262894509.py:15: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("kata")


kata,frekuensi
str,u32
"""jokowi""",1627
"""indonesia""",1078
"""polisi""",1014
"""kasus""",1003
"""lebaran""",971
…,…
"""buka""",654
"""warga""",640
"""juta""",638


### Visualisasi 6: 20 kata terbanyak pada judul

In [14]:
urut = top_kata.sort("frekuensi")
fig6 = go.Figure(go.Bar(
    x=urut["frekuensi"].to_list(), y=urut["kata"].to_list(), orientation="h",
))
fig6.update_layout(
    title="20 Kata Paling Sering Muncul pada Judul Berita",
    xaxis_title="Frekuensi", yaxis_title="Kata", height=600,
)
simpan(fig6, "06_top_kata_judul")
fig6.show()

Tersimpan: output\figures\06_top_kata_judul.html


### Visualisasi 7: Tren bulanan 5 kata teratas

Frekuensi dinormalkan per 1.000 artikel pada bulan yang sama, agar naik-turunnya volume berita tidak menyesatkan.

In [16]:
top5 = top_kata["kata"].head(5).to_list()

# Total artikel per bulan dihitung dengan Polars agar tipe tanggalnya sama dengan kata_bulan
total_bulan = dict(
    df.group_by(pl.col("date").dt.truncate("1mo").alias("bulan"))
    .agg(pl.len().alias("n"))
    .iter_rows()
)

kata_bulan = (
    kata_df.filter(pl.col("kata").is_in(top5))
    .group_by([pl.col("date").dt.truncate("1mo").alias("bulan"), "kata"])
    .agg(pl.len().alias("n"))
    .sort("bulan")
)

fig7 = go.Figure()
for kata in top5:
    bagian = kata_bulan.filter(pl.col("kata") == kata)
    bulan_list = bagian["bulan"].to_list()
    per_seribu = [round(n / total_bulan[b] * 1000, 2) for b, n in zip(bulan_list, bagian["n"].to_list())]
    fig7.add_trace(go.Scatter(x=bulan_list, y=per_seribu, mode="lines+markers", name=kata))
fig7.update_layout(
    title="Tren Bulanan 5 Kata Teratas pada Judul (per 1.000 artikel)",
    xaxis_title="Bulan", yaxis_title="Kemunculan per 1.000 artikel", hovermode="x unified",
)
simpan(fig7, "07_tren_kata_teratas")
fig7.show()

Tersimpan: output\figures\07_tren_kata_teratas.html


## 6. Analisis Spasial: Penyebutan Wilayah pada Judul

Dataset tidak memiliki koordinat, sehingga analisis spasial dilakukan dengan menghitung **penyebutan nama provinsi pada judul**, lalu memetakannya ke titik ibu kota provinsi.

**Keterbatasan:** "Riau" juga menghitung Kepulauan Riau, "Maluku" juga menghitung Maluku Utara, dan "Papua" mencakup semua provinsi Papua. Singkatan (misalnya "Jabar") tidak dihitung.

In [17]:
WILAYAH = [
    # (nama, regex, lintang, bujur)
    ("Aceh", r"aceh", 5.55, 95.32),
    ("Sumatera Utara", r"sumatera utara", 3.59, 98.67),
    ("Sumatera Barat", r"sumatera barat", -0.95, 100.35),
    ("Riau", r"riau", 0.51, 101.45),
    ("Jambi", r"jambi", -1.61, 103.61),
    ("Sumatera Selatan", r"sumatera selatan", -2.98, 104.76),
    ("Bangka Belitung", r"bangka belitung", -2.13, 106.11),
    ("Bengkulu", r"bengkulu", -3.80, 102.26),
    ("Lampung", r"lampung", -5.43, 105.26),
    ("DKI Jakarta", r"jakarta", -6.21, 106.85),
    ("Banten", r"banten", -6.12, 106.15),
    ("Jawa Barat", r"jawa barat", -6.92, 107.61),
    ("Jawa Tengah", r"jawa tengah", -6.97, 110.42),
    ("DI Yogyakarta", r"yogyakarta", -7.80, 110.36),
    ("Jawa Timur", r"jawa timur", -7.25, 112.75),
    ("Bali", r"bali", -8.67, 115.21),
    ("Nusa Tenggara Barat", r"nusa tenggara barat|ntb", -8.58, 116.12),
    ("Nusa Tenggara Timur", r"nusa tenggara timur|ntt", -10.18, 123.61),
    ("Kalimantan Barat", r"kalimantan barat", -0.03, 109.33),
    ("Kalimantan Tengah", r"kalimantan tengah", -2.21, 113.92),
    ("Kalimantan Selatan", r"kalimantan selatan", -3.32, 114.59),
    ("Kalimantan Timur", r"kalimantan timur", -0.50, 117.15),
    ("Kalimantan Utara", r"kalimantan utara", 2.84, 117.37),
    ("Sulawesi Utara", r"sulawesi utara", 1.47, 124.84),
    ("Gorontalo", r"gorontalo", 0.54, 123.06),
    ("Sulawesi Tengah", r"sulawesi tengah", -0.90, 119.87),
    ("Sulawesi Barat", r"sulawesi barat", -2.68, 118.89),
    ("Sulawesi Selatan", r"sulawesi selatan", -5.15, 119.43),
    ("Sulawesi Tenggara", r"sulawesi tenggara", -3.97, 122.51),
    ("Maluku", r"maluku", -3.70, 128.18),
    ("Papua", r"papua", -2.53, 140.72),
]

hitung = df.select([
    pl.col("title").str.contains(rf"(?i)\b(?:{pola})\b").sum().alias(nama)
    for nama, pola, _, _ in WILAYAH
]).row(0)

wilayah_df = pl.DataFrame({
    "wilayah": [w[0] for w in WILAYAH],
    "lintang": [w[2] for w in WILAYAH],
    "bujur": [w[3] for w in WILAYAH],
    "penyebutan": list(hitung),
}).sort("penyebutan", descending=True)
wilayah_df.head(10)

wilayah,lintang,bujur,penyebutan
str,f64,f64,i64
"""DKI Jakarta""",-6.21,106.85,481
"""Bali""",-8.67,115.21,320
"""Papua""",-2.53,140.72,303
"""Lampung""",-5.43,105.26,269
"""Riau""",0.51,101.45,118
"""Aceh""",5.55,95.32,111
"""DI Yogyakarta""",-7.8,110.36,65
"""Nusa Tenggara Timur""",-10.18,123.61,65
"""Banten""",-6.12,106.15,50


### Visualisasi 8: Peta penyebutan wilayah pada judul berita

In [18]:
peta = wilayah_df.filter(pl.col("penyebutan") > 0)
maks = peta["penyebutan"].max()

fig8 = go.Figure(go.Scattergeo(
    lat=peta["lintang"].to_list(), lon=peta["bujur"].to_list(),
    text=[f"{w}: {n:,} judul" for w, n in zip(peta["wilayah"], peta["penyebutan"])],
    hoverinfo="text",
    marker=dict(
        size=[8 + 42 * (n / maks) ** 0.5 for n in peta["penyebutan"]],
        color=peta["penyebutan"].to_list(), colorscale="Reds", showscale=True,
        colorbar_title="Penyebutan", line=dict(width=0.5, color="white"), opacity=0.8,
    ),
))
fig8.update_geos(
    lonaxis_range=[94, 142], lataxis_range=[-12, 8],
    showland=True, landcolor="#f2f2f2", showcountries=True, showocean=True, oceancolor="#e8f1fa",
)
fig8.update_layout(title="Penyebutan Wilayah pada Judul Berita", height=500, margin=dict(l=0, r=0, t=50, b=0))
simpan(fig8, "08_peta_penyebutan_wilayah")
fig8.show()

Tersimpan: output\figures\08_peta_penyebutan_wilayah.html


## 7. Bahan Insight

In [19]:
# Sumber dominan
top_sumber = ringkasan_sumber.row(0, named=True)
total = df.height
print(f"Sumber terbanyak : {top_sumber['source']} ({top_sumber['jumlah_artikel']:,} artikel, "
      f"{top_sumber['jumlah_artikel'] / total * 100:.1f}% dari total)")

# Puncak dan terendah bulanan (tanpa bulan pertama dan terakhir)
if per_bulan.height >= 3:
    penuh = per_bulan.slice(1, per_bulan.height - 2)
    puncak = penuh.sort("jumlah_artikel", descending=True).row(0, named=True)
    rendah = penuh.sort("jumlah_artikel").row(0, named=True)
    print(f"Bulan puncak     : {puncak['bulan']} ({puncak['jumlah_artikel']:,} artikel)")
    print(f"Bulan terendah   : {rendah['bulan']} ({rendah['jumlah_artikel']:,} artikel)")

# Hari tersibuk dan tersepi
hari_total = (
    df.with_columns(pl.col("date").dt.weekday().alias("hari"))
    .group_by("hari").agg(pl.len().alias("n")).sort("n", descending=True)
)
tersibuk = hari_total.row(0, named=True)
tersepi = hari_total.row(-1, named=True)
print(f"Hari tersibuk    : {hari_nama[tersibuk['hari'] - 1]} ({tersibuk['n']:,} artikel)")
print(f"Hari tersepi     : {hari_nama[tersepi['hari'] - 1]} ({tersepi['n']:,} artikel)")

# Panjang artikel (median) per sumber
median_sumber = (
    df.group_by("source").agg(pl.col("panjang_isi").median().alias("median_panjang_isi"))
    .sort("median_panjang_isi", descending=True)
)
print("\nMedian panjang isi per sumber:")
print(median_sumber)

# Kata dan wilayah teratas
print("\n5 kata teratas judul :", top5)
print("5 wilayah teratas    :", wilayah_df.head(5)["wilayah"].to_list())

Sumber terbanyak : kumparan (8,569 artikel, 26.6% dari total)
Bulan puncak     : 2023-03-01 00:00:00 (20,935 artikel)
Bulan terendah   : 2023-03-01 00:00:00 (20,935 artikel)
Hari tersibuk    : Selasa (7,037 artikel)
Hari tersepi     : Minggu (2,984 artikel)

Median panjang isi per sumber:
shape: (7, 2)
┌───────────────┬────────────────────┐
│ source        ┆ median_panjang_isi │
│ ---           ┆ ---                │
│ str           ┆ f64                │
╞═══════════════╪════════════════════╡
│ tempo         ┆ 2785.0             │
│ suara         ┆ 2306.5             │
│ jawapos       ┆ 2235.0             │
│ cnbcindonesia ┆ 2229.0             │
│ cnnindonesia  ┆ 1984.0             │
│ kumparan      ┆ 1904.0             │
│ okezone       ┆ 1852.0             │
└───────────────┴────────────────────┘

5 kata teratas judul : ['jokowi', 'indonesia', 'polisi', 'kasus', 'lebaran']
5 wilayah teratas    : ['DKI Jakarta', 'Bali', 'Papua', 'Lampung', 'Riau']


## 8. Insight Analitik

**Insight 1: Komposisi dan cakupan sumber tidak seimbang**
- Temuan: Dua sumber, Kumparan dan Okezone, menyumbang hampir setengah dari seluruh artikel, dan periode cakupan tiap sumber berbeda.
- Bukti (Visualisasi 1 dan tabel ringkasan sumber): Kumparan 8.569 artikel (26,6%) dan Okezone 7.063 artikel (±21,9%), gabungannya ±48,4% dari 32.273 artikel. JawaPos hanya 1.009 artikel (±3,1%) dan datanya berhenti pada 29 Maret 2023, sedangkan sumber lain sampai 11 April 2023. Tanggal mulai juga berbeda (Tempo 3 Maret, Suara 12 Maret).
- Implikasi: Perbedaan jumlah artikel lebih mencerminkan cakupan *crawler* daripada produktivitas redaksi. Perbandingan antar sumber sebaiknya memakai proporsi atau rata-rata per hari aktif, bukan jumlah mentah.

**Insight 2: Artikel terbit paling banyak di awal pekan dan paling sedikit di akhir pekan**
- Temuan: Pola mingguan konsisten di ketujuh sumber: Senin dan Selasa dominan, akhir pekan paling sepi, dengan tingkat penurunan berbeda per sumber.
- Bukti (Visualisasi 4): Selasa adalah hari tersibuk (7.037 artikel, ±21,8% dari total) dan Minggu tersepi (2.984 artikel, ±9,2%). Senin ditambah Selasa menyumbang 39,9% sampai 44,8% artikel tiap sumber, jauh di atas porsi merata dua hari (±28,6%). Porsi Sabtu ditambah Minggu terendah pada CNBC Indonesia (13,5%) dan tertinggi pada Kumparan (23,1%) dan Tempo (21,7%).
- Implikasi: Volume berita punya siklus mingguan, sehingga data harian perlu dibandingkan pada hari yang sama atau dinormalkan per hari dalam seminggu. Sumber dengan porsi akhir pekan tinggi (Kumparan, Tempo) kemungkinan punya jadwal publikasi atau jenis konten yang lebih santai. Pola ini baru diamati pada ±5 minggu, jadi perlu data lebih panjang untuk memastikan.

**Insight 3: Tempo menulis artikel paling panjang, Okezone paling pendek**
- Temuan: Panjang artikel berbeda jelas antar sumber, menandakan gaya penulisan yang berbeda.
- Bukti (Visualisasi 5 dan sel Bahan Insight): Median panjang isi Tempo 2.785 karakter, sedangkan Okezone 1.852 karakter (Tempo ±1,5 kali lebih panjang). Sumber lain di antaranya: Suara 2.306,5, JawaPos 2.235, CNBC Indonesia 2.229, CNN Indonesia 1.984, Kumparan 1.904.
- Implikasi: Untuk analisis teks (misalnya peringkasan atau klasifikasi), sumber dapat menjadi variabel pengontrol karena panjang dan gaya tulisannya berbeda. Pencampuran semua sumber tanpa penyeimbangan bisa membuat model bias ke gaya Tempo atau Okezone.

**Insight 4: Agenda berita dipengaruhi musim Ramadan dan Lebaran 2023**
- Temuan: Selain topik tetap (politik dan hukum), kata-kata musiman masuk jajaran kata teratas pada judul, dan liputan Lebaran meningkat tajam dari Maret ke April.
- Bukti (Visualisasi 6 dan 7): Kata terbanyak pada judul adalah "jokowi" (1.627 kemunculan), "indonesia" (1.078), "polisi" (1.014), "kasus" (1.003), lalu "lebaran" (971). "harga" (863), "mudik" (814), dan "ramadhan" (797) juga ada pada 20 besar. Setelah dinormalkan per 1.000 artikel, "lebaran" naik dari 18,1 (Maret) menjadi 52,2 (April), sekitar 2,9 kali lipat, sedangkan "indonesia" (33,3 ke 33,6) dan "kasus" (32,2 ke 29,1) relatif stabil.
- Implikasi: Topik berita dipengaruhi kalender keagamaan dan musiman, sehingga analisis topik harus memperhitungkan periode (misalnya mudik dan harga kebutuhan pokok menjelang Lebaran). Kata "harga" kemungkinan terkait kenaikan harga bahan pokok; hal ini perlu dicek dengan membaca contoh judulnya sebelum disimpulkan.

**Insight 5: Liputan terpusat di Jakarta, tetapi beberapa daerah di luar Jawa menonjol**
- Temuan: Penyebutan wilayah pada judul didominasi DKI Jakarta, disusul tiga daerah di luar Jawa yang lebih sering disebut daripada hampir semua provinsi di Jawa.
- Bukti (Visualisasi 8): DKI Jakarta 481 penyebutan, Bali 320, Papua 303, Lampung 269, lalu Riau 118 dan Aceh 111. Provinsi Jawa lain tidak masuk 10 besar kecuali DI Yogyakarta (65) dan Banten (50).
- Implikasi: Jakarta menjadi pusat pemberitaan nasional, tetapi isu daerah tertentu bisa mengangkat sebuah provinsi ke pusat perhatian dalam periode singkat. Penyebab tingginya Bali, Papua, dan Lampung perlu diverifikasi dengan membaca contoh judulnya, karena angka ini belum menjelaskan konteks pemberitaannya.

### Keterbatasan analisis

- Data hasil *scraping*: volume per sumber dan per bulan bias oleh cakupan *crawler*, dan tiap sumber punya rentang tanggal berbeda.
- Rentang waktu efektif hanya ±5 minggu (3 Maret sampai 11 April 2023); terdapat 1 artikel tahun 2018 yang menjadi pencilan waktu. Tren bulanan hanya membandingkan dua bulan, dan April belum lengkap, sehingga kesimpulan temporal bersifat sementara.
- Penyebutan wilayah hanya dari judul dan tidak membedakan konteks. "Papua" mencakup semua provinsi Papua, "Riau" juga menghitung Kepulauan Riau, dan singkatan seperti "Jabar" tidak terhitung.
- Analisis kata hanya pada judul dan belum memakai lematisasi atau pembersihan kata dasar.